In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
output_dir = 'synthetic_workloads'
os.makedirs(output_dir, exist_ok=True)

df = pd.read_parquet('../dataset/df_final.parquet')

In [3]:
MEAN_INTER_ARRIVAL_SECONDS = 150 

NUM_JOBS_PER_SIM = 10000
NUM_SIMULATIONS = 50

In [4]:
for i in range(NUM_SIMULATIONS):
    synthetic = df.sample(n=NUM_JOBS_PER_SIM, replace=True).copy()
    
    synthetic = synthetic.sample(frac=1).reset_index(drop=True)
    
    gaps = np.random.exponential(scale=MEAN_INTER_ARRIVAL_SECONDS, size=NUM_JOBS_PER_SIM)
    synthetic['submit_time'] = np.cumsum(gaps)
    
    overestimation_factor = np.random.uniform(1.2, 5.0, size=NUM_JOBS_PER_SIM)
    
    synthetic['actual_runtime'] = np.maximum(1.0, synthetic['requested_time'] / overestimation_factor)
    
    cols_to_keep = [
        'submit_time', 'requested_processors', 'requested_time', 
        'actual_runtime', 'queue_number', 'think_time'
    ]
    final_workload = synthetic[cols_to_keep]
    
    file_path = os.path.join(output_dir, f'workload_{i+1:02d}.csv')
    final_workload.to_csv(file_path, index=False)

print(f"✅ Successfully generated {NUM_SIMULATIONS} unique workloads in the '{output_dir}/' folder!")

✅ Successfully generated 50 unique workloads in the 'synthetic_workloads/' folder!


In [5]:
tdf = pd.read_csv('../synthetic_workloads/workload_01.csv')

In [7]:
tdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   submit_time           10000 non-null  float64
 1   requested_processors  10000 non-null  int64  
 2   requested_time        10000 non-null  int64  
 3   actual_runtime        10000 non-null  float64
 4   queue_number          10000 non-null  int64  
 5   think_time            10000 non-null  float64
dtypes: float64(3), int64(3)
memory usage: 468.9 KB


In [8]:
tdf.describe()

,submit_time,requested_processors,requested_time,actual_runtime,queue_number,think_time
count,1.000000e+04,10000.000000,1.000000e+04,1.000000e+04,10000.000000,1.000000e+04
mean,7.621015e+05,4.142100,9.692523e+04,3.866792e+04,2.751600,1.466405e+03
std,4.374692e+05,14.310674,1.439681e+06,6.592989e+05,1.315397,7.156885e+04
min,3.859256e+02,1.000000,3.000000e+01,6.034554e+00,1.000000,0.000000e+00
25%,3.826119e+05,1.000000,7.140000e+03,1.844661e+03,2.000000,0.000000e+00
50%,7.684534e+05,1.000000,9.000000e+03,3.492238e+03,2.000000,0.000000e+00
75%,1.140909e+06,1.000000,1.620000e+04,7.775094e+03,3.000000,0.000000e+00
max,1.511696e+06,256.000000,3.600000e+07,2.818834e+07,6.000000,6.629678e+06
